1.- Realiza una investigación científica sobre el filtro Bayer aplicado en el procesamiento digital de imágenes, el porque se aplica y como proviene; de la misma manera realizando código lista las versiones de filtro Bayer en cv2 y aplícalos en tus imágenes seleccionadas.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- FUNCIONES DE PROCESAMIENTO ---

def cargar_imagen(nombre_archivo):
    # Buscamos el archivo en el directorio actual
    ruta = Path(nombre_archivo)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo {nombre_archivo} en la carpeta actual.")
    
    imagen_bgr = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
    if imagen_bgr is None:
        raise ValueError("Error al decodificar la imagen.")
        
    # Convertimos de BGR (OpenCV) a RGB (Matplotlib)
    return cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

def separar_canales(imagen):
    return imagen[:, :, 0], imagen[:, :, 1], imagen[:, :, 2]

def reconstruir_rgb(rojo, verde, azul):
    return np.stack((rojo, verde, azul), axis=2)

def aplicar_patron_bayer(rojo, verde, azul, patron):
    r = np.zeros_like(rojo)
    g = np.zeros_like(verde)
    b = np.zeros_like(azul)

    # Implementación de los 4 tipos de mosaico Bayer
    if patron == "BGGR":
        b[0::2, 0::2] = azul[0::2, 0::2]
        g[0::2, 1::2] = verde[0::2, 1::2]
        g[1::2, 0::2] = verde[1::2, 0::2]
        r[1::2, 1::2] = rojo[1::2, 1::2]
    elif patron == "GBRG":
        g[0::2, 0::2] = verde[0::2, 0::2]
        b[0::2, 1::2] = azul[0::2, 1::2]
        r[1::2, 0::2] = rojo[1::2, 0::2]
        g[1::2, 1::2] = verde[1::2, 1::2]
    elif patron == "GRBG":
        g[0::2, 0::2] = verde[0::2, 0::2]
        r[0::2, 1::2] = rojo[0::2, 1::2]
        b[1::2, 0::2] = azul[1::2, 0::2]
        g[1::2, 1::2] = verde[1::2, 1::2]
    elif patron == "RGGB":
        r[0::2, 0::2] = rojo[0::2, 0::2]
        g[0::2, 1::2] = verde[0::2, 1::2]
        g[1::2, 0::2] = verde[1::2, 0::2]
        b[1::2, 1::2] = azul[1::2, 1::2]

    return reconstruir_rgb(r, g, b)

def mostrar_resultados(imagen_original, resultados):
    figura, ejes = plt.subplots(3, 2, figsize=(12, 12))
    
    # Imagen Original
    ejes[0, 0].imshow(imagen_original)
    ejes[0, 0].set_title("Imagen Original (windows.png)")
    ejes[0, 0].axis("off")
    ejes[0, 1].axis("off") # Espacio libre

    patrones = list(resultados.keys())
    posiciones = [(1, 0), (1, 1), (2, 0), (2, 1)]

    for i, p in enumerate(patrones):
        f, c = posiciones[i]
        ejes[f, c].imshow(resultados[p])
        ejes[f, c].set_title(f"Patrón Bayer: {p}")
        ejes[f, c].axis("off")

    plt.tight_layout()
    plt.show()

# --- EJECUCIÓN PRINCIPAL ---

try:
    # 1. Cargar específicamente windows.png
    imagen = cargar_imagen("windows.png")
    
    # 2. Procesar
    rojo, verde, azul = separar_canales(imagen)
    
    patrones = ["BGGR", "GBRG", "GRBG", "RGGB"]
    resultados = {p: aplicar_patron_bayer(rojo, verde, azul, p) for p in patrones}
    
    # 3. Graficar
    mostrar_resultados(imagen, resultados)
    print("Mosaicos de Bayer para windows.png generados con éxito.")

except Exception as e:
    print(f"Error: {e}")

Error: No se encontró el archivo windows.png en la carpeta actual.
